In [2]:
import pandas as pd
import numpy as np
import sys


In [3]:
#Q2.1
from Risk_Library import first_four_moments

problem2 = pd.read_csv("problem2.csv")
m = first_four_moments(problem2)
print(m)
print(problem2.describe())

                 X
Mean     -0.000346
Variance  0.000485
Skewness  0.114253
Kurtosis  0.968389
                 X
count  1000.000000
mean     -0.000346
std       0.022032
min      -0.082708
25%      -0.014359
50%      -0.000778
75%       0.013657
max       0.087231


# Q2.2
use t since Kurtosis is significantly > 0 while skewness is about 0

In [4]:
#Q 2.3
mu = m.loc["Mean","X"]
sigma = np.sqrt(m.loc["Variance","X"])

from Risk_Library import fit_t_distribution, fit_normal_distribution

data = problem2["X"]
t_nom = fit_t_distribution(data, output= "distribution")
nom = fit_normal_distribution(data, output= "distribution")

q01 = nom.ppf(0.01)
n_below = ( data< q01).sum()
n_expected = 0.01 * len(data)

print(f"Normal 1% quantile = {q01}")
print(f"Observations below quantile = {n_below}")
print(f"Expected observations = {n_expected}")


q01 = t_nom.ppf(0.01)
n_below = ( data< q01).sum()
n_expected = 0.01 * len(data)

print(f"T-Normal 1% quantile = {q01}")
print(f"Observations below quantile = {n_below}")
print(f"Expected observations = {n_expected}")
print("")
print("T-normal has number of observations below 1% quantile closer to the expected number, so it proves t is better")

Normal 1% quantile = -0.05160062097556656
Observations below quantile = 13
Expected observations = 10.0
T-Normal 1% quantile = -0.05539183424986112
Observations below quantile = 9
Expected observations = 10.0

T-normal has number of observations below 1% quantile closer to the expected number, so it proves t is better


In [5]:
#Q3.1

from Risk_Library import calculate_var

t_var = calculate_var(data, 
                      distribution= "t", 
                      alpha= 0.05,
                      seed= 42
                      )

print(t_var)
print(f"T-Normal VaR = {t_var["VaR Absolute"]}")


norm_var = calculate_var(data, 
                      distribution= "normal", 
                      alpha= 0.05,
                      seed= 42
                      )
print(f"Normal VaR = {norm_var["VaR Absolute"]}")

VaR Absolute          0.036102
VaR Diff from Mean    0.035625
dtype: float64
T-Normal VaR = 0.03610246447172258
Normal VaR = 0.03658572011392575


In [6]:
#Q3.2

from Risk_Library import calculate_es

t_es = calculate_es(data, 
                      distribution= "t", 
                      alpha= 0.05,
                      seed= 42
                      )

norm_es = calculate_es(data, 
                      distribution= "normal", 
                      alpha= 0.05,
                      seed= 42
                      )

print(t_es)

print(f"T-Normal ES = {t_es["ES Absolute"]}")
print(f"Normal ES = {norm_es["ES Absolute"]}")

ES Absolute          0.048230
ES Diff from Mean    0.047753
dtype: float64
T-Normal ES = 0.048230230953953686
Normal ES = 0.045792128233980406


# Q3.3

t has fatter tail since ES is larger but have smaller VaR

In [7]:
#Q4.1

problem4 = pd.read_csv("problem4.csv")
print(problem4.head())

from Risk_Library import ew_covariance_matrix

ew_cor_matrix = ew_covariance_matrix(problem4,
                                  lambda_= 0.94,
                                  method= "correlation"
                                  )

print(ew_cor_matrix)


         x1        x2        x3
0  0.034246  0.121262 -0.115528
1  0.131169  0.192027  0.043622
2  0.125515 -0.108881 -0.004787
3  0.157331  0.033694  0.109034
4 -0.119847 -0.249491 -0.234808
          x1        x2        x3
x1  1.000000  0.711329  0.807175
x2  0.711329  1.000000  0.713020
x3  0.807175  0.713020  1.000000


In [8]:
#Q4.2



ew_cov_matrix = ew_covariance_matrix(problem4,
                                     lambda_= 0.97,
                                     method= "covariance"
                                     )
print(ew_cov_matrix)

          x1        x2        x3
x1  0.015379  0.014257  0.014839
x2  0.014257  0.035517  0.021057
x3  0.014839  0.021057  0.027813


In [9]:
#Q4.3

combined_ew_cov_matrix = ew_covariance_matrix(problem4,
                                     lambda_= 0.97,
                                     method= "covariance",
                                     lambda_corr= 0.94
                                     )
print(combined_ew_cov_matrix)

          x1        x2        x3
x1  0.015379  0.016625  0.016694
x2  0.016625  0.035517  0.022410
x3  0.016694  0.022410  0.027813


In [10]:
#Q4.4


#???????????????????????

In [11]:
#Q5.1
problem5 = pd.read_csv("problem5.csv")
print(problem5.head())


from Risk_Library import covariance_matrix

cov_matrix = covariance_matrix(problem5, 
                               missing_strategy= "pairwise",
                               )

print(cov_matrix)

         x1        x2        x3        x4        x5
0  0.378172  0.508313       NaN  0.610538       NaN
1  0.858478  0.758142       NaN       NaN       NaN
2 -0.218954 -0.015334 -0.261880       NaN -0.230148
3 -0.370442 -0.424702 -0.535312       NaN -0.226233
4  0.717732  0.664195       NaN       NaN  0.914107
          x1        x2        x3        x4        x5
x1  1.470484  1.454214  0.877269  1.903226  1.444361
x2  1.454214  1.252078  0.539548  1.621918  1.237877
x3  0.877269  0.539548  1.272425  1.171959  1.091912
x4  1.903226  1.621918  1.171959  1.814469  1.589729
x5  1.444361  1.237877  1.091912  1.589729  1.396186


In [12]:
#Q5.2

from Risk_Library import classify_psd

results = classify_psd(cov_matrix)
print(results)


{'classification': 'non-PSD', 'summary': 'non-PSD: 2 eigenvalue(s) below -threshold (6.79e-08): [-0.3102, -0.1332].', 'is_symmetric': True, 'max_asymmetry': 0.0, 'symmetrized': False, 'eigenvalues': array([-0.31024286, -0.13323183,  0.02797828,  0.83443367,  6.78670573]), 'adjusted_eigenvalues': array([-0.31024286, -0.13323183,  0.02797828,  0.83443367,  6.78670573]), 'threshold': 6.786805732478079e-08, 'small_negative_indices': array([], dtype=int64), 'small_negative_values': array([], dtype=float64), 'negative_indices': array([0, 1]), 'negative_values': array([-0.31024286, -0.13323183]), 'min_eigenvalue': -0.31024286045858995, 'max_eigenvalue': 6.786705732478078}


In [13]:
#5.3

from Risk_Library import fix_psd_matrix

fix_cov_matirx = fix_psd_matrix(cov_matrix, method="higham")
print(fix_cov_matirx)


          x1        x2        x3        x4        x5
x1  1.470484  1.332361  0.884378  1.627602  1.399556
x2  1.332361  1.252078  0.619028  1.450604  1.214450
x3  0.884378  0.619028  1.272425  1.076846  1.059658
x4  1.627602  1.450604  1.076846  1.814469  1.577928
x5  1.399556  1.214450  1.059658  1.577928  1.396186


In [14]:
#5.4

#??????????????????????/

In [15]:
#Q6.1

problem6 = pd.read_csv("problem6.csv")
print(problem6.head())


           x1          x2          x3        Date
0  102.975983  101.234230  101.390965  2025-01-01
1  102.553769  102.056711  103.589863  2025-01-02
2  102.362551  100.605815   99.204933  2025-01-03
3  104.522707  100.479907  101.798573  2025-01-04
4  103.682209  100.637430  101.599604  2025-01-05


In [16]:
from Risk_Library import return_calculate
return_matrix = return_calculate(problem6,
                                 method="arithmetic",
                                 date_column="Date")

print(return_matrix.head())

         Date        x1        x2        x3
0  2025-01-02 -0.004100  0.008125  0.021687
1  2025-01-03 -0.001865 -0.014217 -0.042330
2  2025-01-04  0.021103 -0.001252  0.026144
3  2025-01-05 -0.008041  0.001568 -0.001955
4  2025-01-06 -0.003674 -0.006858 -0.027222


In [17]:
#6.1
return_cols = return_matrix.columns.drop("Date")
demeaned_returns = return_matrix.copy()

demeaned_returns[return_cols] = return_matrix[return_cols] - return_matrix[return_cols].mean()

print(demeaned_returns)

x1_fit = fit_t_distribution(demeaned_returns["x1"],
                            output= "params")
x2_fit = fit_t_distribution(demeaned_returns["x2"],
                            output= "params")
x3_fit = fit_t_distribution(demeaned_returns["x3"],
                            output= "params")

print(x1_fit)
print(x2_fit)
print(x3_fit)

           Date        x1        x2        x3
0    2025-01-02 -0.003377  0.009144  0.022378
1    2025-01-03 -0.001141 -0.013197 -0.041639
2    2025-01-04  0.021827 -0.000232  0.026835
3    2025-01-05 -0.007318  0.002587 -0.001264
4    2025-01-06 -0.002950 -0.005839 -0.026531
..          ...       ...       ...       ...
244  2025-09-03 -0.016598 -0.028312 -0.001771
245  2025-09-04 -0.004847  0.000280 -0.014978
246  2025-09-05  0.016054  0.006029  0.009522
247  2025-09-06  0.017341  0.010856  0.026284
248  2025-09-07  0.021014  0.023501  0.031127

[249 rows x 4 columns]
mu      -0.000479
sigma    0.012908
nu       4.729831
dtype: float64
mu      -0.000044
sigma    0.009058
nu       6.766945
dtype: float64
mu        0.000075
sigma     0.017063
nu       39.864293
dtype: float64


In [26]:
#6.2

from Risk_Library import simulate_copula


return_data = demeaned_returns[return_cols]

results = simulate_copula(return_data, seed=42, output= "both")

print(results[0])
print(results[1]["correlation"])



             x1        x2        x3
0      0.003689 -0.012576  0.030401
1     -0.015553  0.017060  0.004917
2      0.010070  0.021225  0.010145
3      0.012995 -0.002839 -0.014064
4     -0.034007  0.004807 -0.008416
...         ...       ...       ...
99995  0.013876  0.003683  0.014276
99996  0.009865  0.007791  0.004173
99997  0.022931  0.012682  0.008984
99998 -0.002566  0.009045  0.009341
99999  0.008650  0.005114  0.013134

[100000 rows x 3 columns]
          x1        x2        x3
x1  1.000000  0.464893  0.437596
x2  0.464893  1.000000  0.519012
x3  0.437596  0.519012  1.000000


In [29]:
#6.3  VaR and ES (5% alpha) for each stock, in $
from Risk_Library import price_from_returns

alpha = 0.05
shares = 100   # we own 100 shares of each stock

# Simulated one-day returns from the Gaussian copula in #6.2
# (100,000 scenarios x 3 stocks; t marginals fitted to the de-meaned returns).
simulated_returns = results[0]

# Today's price = the last row of problem6.csv. Risk is measured on what we hold now.
current_prices = problem6[return_cols].iloc[-1]
holding_values = shares * current_prices

# Turn each simulated arithmetic return into tomorrow's price:
#   P_t = P_(t-1) * (1 + r)          ("one_step": each row is a separate scenario)
simulated_prices = price_from_returns(simulated_returns,
                                      start_price=current_prices,
                                      method="arithmetic")

# Dollar P&L per scenario = shares * (tomorrow's price - today's price)
simulated_pnl = shares * (simulated_prices - current_prices)


def var_es_from_pnl(pnl, alpha):
    """VaR and ES from simulated P&L, both reported as positive dollar losses."""
    cutoff = np.quantile(pnl, alpha)        # the 5% worst-case P&L
    var = -cutoff                           # VaR = loss at that cutoff
    es = -pnl[pnl <= cutoff].mean()         # ES = average loss beyond the cutoff
    return var, es


# Apply it to each stock's own P&L column separately
stock_risk = {}
for stock in return_cols:
    var, es = var_es_from_pnl(simulated_pnl[stock].values, alpha)
    stock_risk[stock] = {"Holding Value ($)": holding_values[stock],
                         "VaR 5% ($)": var,
                         "ES 5% ($)": es}
stock_risk = pd.DataFrame(stock_risk).T

print(stock_risk)

    Holding Value ($)  VaR 5% ($)   ES 5% ($)
x1        8306.321359  225.189391  323.942672
x2        7740.486139  134.445221  183.999357
x3        8215.614780  235.496423  299.404521


In [30]:
#6.4  VaR and ES (5% alpha) for the total portfolio, in $

# Portfolio P&L in each scenario = sum of the 3 stocks' P&L in that SAME scenario.
# Summing scenario-by-scenario keeps the copula's dependence between the stocks.
total_pnl = simulated_pnl.sum(axis=1).values
total_var, total_es = var_es_from_pnl(total_pnl, alpha)

print(f"Portfolio value  = ${holding_values.sum():,.2f}")
print(f"Portfolio VaR 5% = ${total_var:,.2f}")
print(f"Portfolio ES 5%  = ${total_es:,.2f}")

# Diversification: the portfolio VaR is less than the sum of the stand-alone VaRs,
# because the stocks are not perfectly correlated (they don't all crash together).
print(f"Sum of stock VaRs = ${stock_risk['VaR 5% ($)'].sum():,.2f}")

# Cross-check with the library: calculate_portfolio_var_es runs the same model
# (t marginals + Gaussian copula, same seed), so the numbers should match exactly.
from Risk_Library import calculate_portfolio_var_es

portfolio = pd.DataFrame({"Stock": list(return_cols),
                          "Holding": shares,
                          "Starting Price": current_prices.values,
                          "Distribution": "T"})
print(calculate_portfolio_var_es(portfolio, return_data, alpha=alpha, seed=42))

Portfolio value  = $24,262.42
Portfolio VaR 5% = $486.08
Portfolio ES 5%  = $644.91
Sum of stock VaRs = $595.13
   Stock       VaR95        ES95  VaR95_Pct  ES95_Pct
0     x1  225.189391  323.942672   0.027111  0.039000
1     x2  134.445221  183.999357   0.017369  0.023771
2     x3  235.496423  299.404521   0.028664  0.036443
3  Total  486.080391  644.911011   0.020034  0.026581
